In [ ]:
# import os
# import pandas as pd
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoProcessor, MoonshineForConditionalGeneration, AutoTokenizer
# import librosa
# import jiwer
# import re
# from tqdm import tqdm

# # --- 1. 設定 ---
# FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
# TRAIN_CSV = "data/train_data.csv"
# VALID_CSV = "data/valid_data.csv"
# MODEL_NAME = "UsefulSensors/moonshine-tiny"
# OUTPUT_MODEL_DIR = "./moonshine-tiny-finetuned-fsc"
# LOG_FILE_PATH = "training_log.csv"

# # DataLoader 相關參數
# BATCH_SIZE = 32
# NUM_WORKERS = 4

# # 訓練參數
# LEARNING_RATE = 1e-4
# EPOCHS = 1 
# # -----------------

# class FluentSpeechDataset(Dataset):
#     def __init__(self, csv_path, fsc_base_path):
#         self.df = pd.read_csv(csv_path)
#         self.fsc_base_path = fsc_base_path

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         relative_path = row['path']
#         reference_text = row['transcription']
#         full_audio_path = os.path.join(self.fsc_base_path, relative_path)
#         audio_array, _ = librosa.load(full_audio_path, sr=16000)
#         return {"audio": audio_array, "text": reference_text}

# def normalize_text(text):
#     if text is None: return ""
#     text = text.lower()
#     text = re.sub(r"[^\w\s]", "", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# class DataCollatorSpeechSeq2Seq:
#     def __init__(self, processor):
#         self.processor = processor
#         self.tokenizer = processor.tokenizer
    
#     def __call__(self, features):
#         input_audios = [f["audio"] for f in features]
#         label_texts = [normalize_text(f["text"]) for f in features]
        
#         # 處理音頻輸入
#         inputs = self.processor(
#             audio=input_audios, 
#             sampling_rate=16000, 
#             return_tensors="pt", 
#             padding=True
#         )
        
#         # 處理文字標籤
#         labels = self.tokenizer(
#             label_texts,
#             padding=True,
#             truncation=True,
#             return_tensors="pt"
#         )
        
#         # ✅ 將 PAD token 替換為 -100（訓練時會被忽略）
#         labels_input_ids = labels.input_ids.clone()
#         pad_token_id = self.tokenizer.pad_token_id
#         if pad_token_id is not None:
#             labels_input_ids[labels_input_ids == pad_token_id] = -100
        
#         # 只保留 tensor 類型的數據
#         batch = {}
#         for key, value in inputs.items():
#             if isinstance(value, torch.Tensor):
#                 batch[key] = value
        
#         batch["labels"] = labels_input_ids  # 使用處理過的 labels
        
#         return batch

# def run_validation(model, processor, valid_loader, device):
#     print("\nRunning validation...")
#     model.eval()
#     total_val_loss = 0.0
#     all_references = []
#     all_hypotheses = []

#     with torch.no_grad():
#         for batch_idx, batch in enumerate(tqdm(valid_loader, desc="Validation")):
#             batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
#                     for k, v in batch.items()}
            
#             outputs = model(**batch)
#             total_val_loss += outputs.loss.item()
            
#             # 檢測輸入 key
#             input_key = None
#             for possible_key in ['input_features', 'input_values', 'input_ids']:
#                 if possible_key in batch:
#                     input_key = possible_key
#                     break
            
#             if input_key is None:
#                 print("❌ 錯誤：找不到有效的輸入 key")
#                 print(f"可用的 keys: {list(batch.keys())}")
#                 continue
            
#             # ✅ 使用更嚴格的 generation 參數
#             try:
#                 generated_ids = model.generate(
#                     **{input_key: batch[input_key]},
                    
#                     # === 長度控制 ===
#                     max_length=448,                    # Moonshine 標準長度
#                     max_new_tokens=None,               # 不使用 max_new_tokens
#                     min_length=1,
                    
#                     # === 解碼策略 ===
#                     num_beams=1,                       # Greedy decoding（最快）
#                     do_sample=False,                   # 不使用採樣
                    
#                     # === 停止條件 ===
#                     early_stopping=True,               # 遇到 EOS 立即停止
#                     eos_token_id=processor.tokenizer.eos_token_id,
#                     pad_token_id=processor.tokenizer.pad_token_id,
                    
#                     # === 🔥 防止重複的關鍵參數 ===
#                     no_repeat_ngram_size=4,            # 禁止重複 4-gram
#                     repetition_penalty=1.5,            # 強力懲罰重複（提高到 1.5）
                    
#                     # === 長度懲罰 ===
#                     length_penalty=0.8,                # 稍微偏好較短的輸出
                    
#                     # === 輸出限制 ===
#                     output_scores=False,               # 不需要 scores
#                     return_dict_in_generate=False,     # 只返回 token IDs
#                 )
#             except Exception as e:
#                 print(f"❌ Generation 錯誤: {e}")
#                 print(f"Batch keys: {list(batch.keys())}")
#                 print(f"Input shape: {batch[input_key].shape if input_key in batch else 'N/A'}")
#                 continue
            
#             # Decode
#             hypotheses = processor.batch_decode(generated_ids, skip_special_tokens=True)
#             hypotheses = [normalize_text(h) for h in hypotheses]
            
#             # 處理 references
#             references = []
#             pad_token_id = processor.tokenizer.pad_token_id
            
#             for label_ids in batch["labels"]:
#                 if pad_token_id is not None:
#                     valid_ids = label_ids[(label_ids != -100) & (label_ids != pad_token_id)]
#                 else:
#                     valid_ids = label_ids[label_ids != -100]
                
#                 ref_text = processor.tokenizer.decode(valid_ids, skip_special_tokens=True)
#                 references.append(normalize_text(ref_text))
            
#             all_hypotheses.extend(hypotheses)
#             all_references.extend(references)
            
#             # 調試輸出
#             if batch_idx == 0:
#                 print("\n" + "="*70)
#                 print("📋 驗證樣本檢查（前 3 個）:")
#                 print("="*70)
#                 for i in range(min(3, len(references))):
#                     print(f"\n樣本 {i+1}:")
#                     print(f"  REF: '{references[i]}'")
#                     print(f"  REF 長度: {len(references[i].split())} 詞")
                    
#                     # 截斷顯示
#                     hyp_display = hypotheses[i] if len(hypotheses[i]) < 120 else hypotheses[i][:120] + "..."
#                     print(f"  HYP: '{hyp_display}'")
#                     print(f"  HYP 長度: {len(hypotheses[i].split())} 詞")
                    
#                     # 檢查是否有重複
#                     words = hypotheses[i].split()
#                     if len(words) > 5:
#                         unique_ratio = len(set(words)) / len(words)
#                         print(f"  唯一詞比例: {unique_ratio:.2%}")
#                         if unique_ratio < 0.5:
#                             print(f"  ⚠️  警告：重複率過高！")
                    
#                     if references[i] and hypotheses[i]:
#                         try:
#                             sample_wer = jiwer.wer([references[i]], [hypotheses[i]])
#                             print(f"  WER: {sample_wer*100:.2f}%")
#                         except:
#                             print(f"  WER: 計算失敗")
                
#                 # ✅ 打印一個完整的 generated_ids 檢查
#                 if len(generated_ids) > 0:
#                     print(f"\n生成的 token IDs (第一個樣本):")
#                     first_ids = generated_ids[0][:50]  # 只看前 50 個 token
#                     print(f"  IDs: {first_ids.tolist() if hasattr(first_ids, 'tolist') else first_ids}")
#                     print(f"  總長度: {len(generated_ids[0])}")
                    
#                     # 檢查是否有 EOS token
#                     eos_id = processor.tokenizer.eos_token_id
#                     if eos_id in generated_ids[0]:
#                         eos_position = (generated_ids[0] == eos_id).nonzero(as_tuple=True)[0][0]
#                         print(f"  EOS token 位置: {eos_position}")
#                     else:
#                         print(f"  ⚠️  警告：沒有找到 EOS token！")
                
#                 print("="*70 + "\n")

#     avg_val_loss = total_val_loss / len(valid_loader) if len(valid_loader) > 0 else 0
    
#     valid_pairs = [(ref, hyp) for ref, hyp in zip(all_references, all_hypotheses) 
#                    if ref.strip() and hyp.strip()]
    
#     if valid_pairs:
#         filtered_references, filtered_hypotheses = zip(*valid_pairs)
#         val_wer = jiwer.wer(list(filtered_references), list(filtered_hypotheses))
        
#         print(f"\n📊 驗證統計:")
#         print(f"  總樣本數: {len(all_references)}")
#         print(f"  有效樣本數: {len(valid_pairs)}")
        
#         # ✅ 計算平均生成長度
#         avg_hyp_len = sum(len(h.split()) for h in filtered_hypotheses) / len(filtered_hypotheses)
#         avg_ref_len = sum(len(r.split()) for r in filtered_references) / len(filtered_references)
#         print(f"  平均生成長度: {avg_hyp_len:.1f} 詞")
#         print(f"  平均參考長度: {avg_ref_len:.1f} 詞")
#         print(f"  長度比例: {avg_hyp_len/avg_ref_len if avg_ref_len > 0 else 0:.2f}x")
        
#         if avg_hyp_len / avg_ref_len > 3:
#             print(f"  ⚠️  警告：生成長度遠超參考長度，可能有問題！")
            
#     else:
#         print("⚠️  警告：沒有有效的樣本對可以計算 WER！")
#         val_wer = 1.0
    
#     model.train()
#     return avg_val_loss, val_wer


# def train_fsc():
#     print(f"--- 開始微調 {MODEL_NAME} ---")
#     train_csv_path = os.path.join(FSC_DATASET_PATH, TRAIN_CSV)
#     valid_csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

#     # 1. 載入模型和 Processor
#     print("正在載入模型...")
#     processor = AutoProcessor.from_pretrained(MODEL_NAME)
    
#     # ✅ 詳細的調試信息
#     print(f"Processor 類型: {type(processor)}")
#     print(f"Tokenizer 類型: {type(processor.tokenizer)}")
#     print(f"原始 pad_token: {processor.tokenizer.pad_token}")
#     print(f"原始 eos_token: {processor.tokenizer.eos_token}")
    
#     # ✅ 針對 Wav2Vec2Processor 的特殊處理
#     tokenizer = processor.tokenizer
    
#     # 檢查是否有 pad_token，如果沒有就添加
#     if tokenizer.pad_token is None:
#         # 嘗試使用 eos_token
#         if tokenizer.eos_token is not None:
#             tokenizer.pad_token = tokenizer.eos_token
#         # 如果沒有 eos_token，檢查是否有 unk_token
#         elif tokenizer.unk_token is not None:
#             tokenizer.pad_token = tokenizer.unk_token
#         # 如果都沒有，添加一個新的 special token
#         else:
#             special_tokens_dict = {'pad_token': '[PAD]'}
#             num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
#             print(f"✅ 添加了 {num_added_tokens} 個新的 special token")
    
#     print(f"設定後 pad_token: {tokenizer.pad_token}")
#     print(f"設定後 pad_token_id: {tokenizer.pad_token_id}")
    
#     model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
#     print(f"\n{'='*60}")
#     print("🔍 模型 Generation Config 檢查")
#     print(f"{'='*60}")
    
#     # 打印原始配置
#     gen_config = model.generation_config
#     print(f"原始 max_length: {gen_config.max_length if hasattr(gen_config, 'max_length') else 'Not set'}")
#     print(f"原始 max_new_tokens: {gen_config.max_new_tokens if hasattr(gen_config, 'max_new_tokens') else 'Not set'}")
#     print(f"原始 eos_token_id: {gen_config.eos_token_id if hasattr(gen_config, 'eos_token_id') else 'Not set'}")
#     print(f"原始 pad_token_id: {gen_config.pad_token_id if hasattr(gen_config, 'pad_token_id') else 'Not set'}")
    
#     # ✅ 設定正確的 generation config
#     # model.generation_config.max_length = 194  # Moonshine 的標準設置
#     model.generation_config.pad_token_id = tokenizer.pad_token_id
#     # model.generation_config.eos_token_id = tokenizer.eos_token_id
#     model.generation_config.decoder_start_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id else tokenizer.eos_token_id
    
#     print(f"\n設定後 max_length: {model.generation_config.max_length}")
#     print(f"設定後 eos_token_id: {model.generation_config.eos_token_id}")
#     print(f"{'='*60}\n")
    
#     # ✅ 如果添加了新的 token，需要調整模型的 embedding 層
#     if tokenizer.pad_token_id is not None:
#         model.config.pad_token_id = tokenizer.pad_token_id
#         # 如果添加了新 token，需要 resize embeddings
#         model.resize_token_embeddings(len(tokenizer))
    
#     # ✅ 設定 device 並移動模型
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"使用設備: {device}")
    
#     model = model.to(device)
    
#     # 驗證模型是否在正確的設備上
#     print(f"模型設備: {next(model.parameters()).device}")
    
#     optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

#     # 2. 建立 Dataset 和 DataLoader
#     train_dataset = FluentSpeechDataset(train_csv_path, FSC_DATASET_PATH)
#     valid_dataset = FluentSpeechDataset(valid_csv_path, FSC_DATASET_PATH)
    
#     data_collator = DataCollatorSpeechSeq2Seq(processor)
    
#     train_loader = DataLoader(
#         train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
#         num_workers=NUM_WORKERS, collate_fn=data_collator
#     )
#     valid_loader = DataLoader(
#         valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
#         num_workers=NUM_WORKERS, collate_fn=data_collator
#     )
    
#     print(f"將在 {len(train_dataset)} 筆訓練資料上訓練 {EPOCHS} 個 epoch...")
    
#     best_val_loss = float('inf')
#     loss_history = []
    
#     for epoch in range(EPOCHS):
#         print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
#         model.train()
#         total_train_loss = 0.0
        
#         for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")):
#             # ✅ 只移動 tensor 到 GPU
#             batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
#                     for k, v in batch.items()}
            
#             optimizer.zero_grad()
#             outputs = model(**batch)
#             loss = outputs.loss
#             loss.backward()
#             optimizer.step()
            
#             total_train_loss += loss.item()
            
#             # ✅ 可選：每 100 步打印一次進度
#             if step % 100 == 0 and step > 0:
#                 print(f"\n  Step {step}/{len(train_loader)}, Current Loss: {loss.item():.4f}")

#         avg_train_loss = total_train_loss / len(train_loader)
#         avg_val_loss, val_wer = run_validation(model, processor, valid_loader, device)
        
#         print(f"\n--- Epoch {epoch + 1} 總結 ---")
#         print(f"  平均訓練 Loss: {avg_train_loss:.4f}")
#         print(f"  驗證 Loss: {avg_val_loss:.4f}")
#         print(f"  驗證 WER: {val_wer * 100:.2f} %")
        
#         epoch_data = {
#             'epoch': epoch + 1, 'train_loss': avg_train_loss,
#             'validation_loss': avg_val_loss, 'validation_wer': val_wer
#         }
#         loss_history.append(epoch_data)
        
#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             print(f"❗️ 新的最佳驗證 Loss。正在儲存模型到 {OUTPUT_MODEL_DIR}")
#             model.save_pretrained(OUTPUT_MODEL_DIR)
#             processor.save_pretrained(OUTPUT_MODEL_DIR)
#         else:
#             print("驗證 Loss 未改善。")

#     print("\n--- 訓練完成 ---")
#     df_log = pd.DataFrame(loss_history)
#     df_log.set_index('epoch', inplace=True)
#     df_log.to_csv(LOG_FILE_PATH)
#     print(f"紀錄已儲存於 {LOG_FILE_PATH}")





In [5]:
if __name__ == "__main__":
    train_fsc()

--- 開始微調 UsefulSensors/moonshine-tiny ---
正在載入模型...
Processor 類型: <class 'transformers.models.wav2vec2.processing_wav2vec2.Wav2Vec2Processor'>
Tokenizer 類型: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
原始 pad_token: None
原始 eos_token: None
✅ 添加了 1 個新的 special token
設定後 pad_token: [PAD]
設定後 pad_token_id: 32768

🔍 模型 Generation Config 檢查
原始 max_length: 194
原始 max_new_tokens: None
原始 eos_token_id: 2
原始 pad_token_id: 2

設定後 max_length: 194
設定後 eos_token_id: 2

使用設備: cuda
模型設備: cuda:0
將在 23132 筆訓練資料上訓練 1 個 epoch...

--- Epoch 1 / 1 ---


Training Epoch 1:   0%|          | 0/723 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Training Epoch 1:  14%|█▍        | 102/723 [00:11<00:56, 11.06it/s]


  Step 100/723, Current Loss: 0.5315


Training Epoch 1:  28%|██▊       | 202/723 [00:20<00:48, 10.80it/s]


  Step 200/723, Current Loss: 0.3150


...


  Step 700/723, Current Loss: 0.2964


Training Epoch 1: 100%|██████████| 723/723 [01:08<00:00, 10.50it/s]



Running validation...


Validation:   0%|          | 0/98 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (194). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
Validation:   


📋 驗證樣本檢查（前 3 個）:

樣本 1:
  REF: 'turn on the lights'
  REF 長度: 4 詞
  HYP: 'on lights in theroom up the down the off the loud it on theheter on themom turn the on the on thelighten on theometromet...'
  HYP 長度: 37 詞
  唯一詞比例: 64.86%
  WER: 850.00%

樣本 2:
  REF: 'turn off the lights'
  REF 長度: 4 詞
  HYP: 'off lights in theroom up the on thehoutherlighten down theighiouteunovw loudzingittingoohmmutofm0ghuneonesomoutsunoffood...'
  HYP 長度: 31 詞
  唯一詞比例: 100.00%
  WER: 750.00%

樣本 3:
  REF: 'change language'
  REF 長度: 2 詞
  HYP: 'language kan switch tolexchangeanguageuteuneuouoiooooghancheanglishngluueujjonesonyujewizdjunesuyomuchinchlanguagemusic ...'
  HYP 長度: 26 詞
  唯一詞比例: 96.15%
  WER: 1300.00%

生成的 token IDs (第一個樣本):
  IDs: [1, 373, 26068, 297, 278, 8345, 701, 278, 1623, 278, 1283, 278, 22526, 372, 373, 278, 29882, 1308, 373, 278, 29885, 290, 2507, 278, 373, 278, 373, 278, 4366, 264, 373, 278, 3297, 29878, 8328, 608, 373, 278, 4317, 24826, 373, 278, 18902, 1300, 3011, 16946, 2770, 680, 

In [17]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MoonshineForConditionalGeneration, AutoTokenizer
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
TRAIN_CSV = "data/train_data.csv"
VALID_CSV = "data/valid_data.csv"
MODEL_NAME = "UsefulSensors/moonshine-tiny"
OUTPUT_MODEL_DIR = "./moonshine-tiny-finetuned-fsc-simplified"
LOG_FILE_PATH = "training_log_simplified.csv"

# DataLoader 相關參數
BATCH_SIZE = 32
NUM_WORKERS = 4 # 若環境支援多執行緒，可設為 > 0

# 訓練參數
LEARNING_RATE = 1e-5
EPOCHS = 50
# -----------------

class FluentSpeechDataset(Dataset):
    def __init__(self, csv_path, fsc_base_path):
        self.df = pd.read_csv(csv_path)
        self.fsc_base_path = fsc_base_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['path']
        reference_text = row['transcription']
        full_audio_path = os.path.join(self.fsc_base_path, relative_path)
        audio_array, _ = librosa.load(full_audio_path, sr=16000)
        return {"audio": audio_array, "text": reference_text}

def normalize_text(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class DataCollatorSpeechSeq2Seq:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
    
    def __call__(self, features):
        input_audios = [f["audio"] for f in features]
        label_texts = [normalize_text(f["text"]) for f in features]
        
        # 處理音頻輸入 (會自動 padding)
        inputs = self.processor(
            audio=input_audios, 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding=True
        )
        
        # 處理文字標籤 (會自動 padding)
        labels = self.tokenizer(
            label_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        
        # ✅ 將 PAD token 替換為 -100，這樣在計算 loss 時會被忽略
        labels_input_ids = labels.input_ids.clone()
        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels_input_ids[labels_input_ids == pad_token_id] = -100
        
        batch = {}
        for key, value in inputs.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value
        
        batch["labels"] = labels_input_ids
        
        return batch

def run_validation(model, processor, valid_loader, device):
    print("\nRunning validation...")
    model.eval()
    total_val_loss = 0.0
    all_references = []
    all_hypotheses = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(valid_loader, desc="Validation")):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # 計算 Loss
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
            
            # 生成預測文字 (使用模型的預設 generation config)
            generated_ids = model.generate(batch["input_values"])
            hypotheses = processor.batch_decode(generated_ids, skip_special_tokens=True)
            
            # 解碼參考文字
            labels = batch["labels"]
            labels[labels == -100] = processor.tokenizer.pad_token_id
            references = processor.batch_decode(labels, skip_special_tokens=True)

            # 正規化文字
            batch_hypotheses = [normalize_text(h) for h in hypotheses]
            batch_references = [normalize_text(r) for r in references]

            # ✅ 在第一個 batch 結束時，印出前 3 筆結果以供檢查
            if batch_idx == 0:
                print("\n" + "="*70)
                print("📋 驗證樣本檢查（第一個 batch 的前 3 筆）:")
                print("="*70)
                for i in range(min(3, len(batch_references))):
                    ref = batch_references[i]
                    hyp = batch_hypotheses[i]
                    print(f"\n樣本 {i+1}:")
                    print(f"  REF: '{ref}'")
                    print(f"  HYP: '{hyp}'")
                    # 計算單一樣本的 WER
                    if ref and hyp:
                        try:
                            sample_wer = jiwer.wer([ref], [hyp])
                            print(f"  WER: {sample_wer*100:.2f}%")
                        except Exception:
                            print("  WER: 計算失敗")
                print("="*70 + "\n")


            all_hypotheses.extend(batch_hypotheses)
            all_references.extend(batch_references)

    avg_val_loss = total_val_loss / len(valid_loader)
    
    # 過濾掉空的參考或預測，避免 jiwer 出錯
    valid_pairs = [(ref, hyp) for ref, hyp in zip(all_references, all_hypotheses) if ref and hyp]
    if not valid_pairs:
        print("警告：沒有有效的樣本對可以計算 WER！")
        val_wer = 1.0
    else:
        filtered_references, filtered_hypotheses = zip(*valid_pairs)
        val_wer = jiwer.wer(list(filtered_references), list(filtered_hypotheses))
    
    model.train()
    return avg_val_loss, val_wer

def train_fsc():
    print(f"--- 開始微調 {MODEL_NAME} ---")
    train_csv_path = os.path.join(FSC_DATASET_PATH, TRAIN_CSV)
    valid_csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

    # 1. 載入模型和 Processor
    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
    tokenizer = processor.tokenizer
   
    
    # --- 關鍵修改區塊 ---
    # ✅ 1. 只設定 [PAD] token，以應對 DataLoader 的批次處理需求
    # 我們需要一個 padding token，但 tokenizer 預設沒有
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '</s>'})
        print(f"✅ 添加了新的 [PAD] token。Tokenizer size: {len(tokenizer)}")
        # 調整模型的 embedding 層以適應新的 token
        model.resize_token_embeddings(len(tokenizer))

    # ✅ 2. 將新的 pad_token_id 同步到模型設定中
    # 這樣模型在內部處理時才知道哪個 ID 是 padding
    model.config.pad_token_id = model.generation_config.eos_token_id
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
    print(f"設定 pad_token 為 eos_token: {tokenizer.pad_token} (ID: {model.generation_config.eos_token_id})")
    # ❗️ 3. 不再對 EOS token 做任何設定
    # 我們完全信任模型預載入的 generation_config，其中 eos_token_id 預設為 2
    print(f"使用模型預設的 eos_token_id: {model.generation_config.eos_token_id}")
    print(f"使用 tokenizer 新設定的 pad_token_id: {tokenizer.pad_token_id}")
    # --- 修改結束 ---
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"使用設備: {device}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    # 2. 建立 Dataset 和 DataLoader
    train_dataset = FluentSpeechDataset(train_csv_path, FSC_DATASET_PATH)
    valid_dataset = FluentSpeechDataset(valid_csv_path, FSC_DATASET_PATH)
    
    data_collator = DataCollatorSpeechSeq2Seq(processor)
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    
    print(f"將在 {len(train_dataset)} 筆訓練資料上訓練 {EPOCHS} 個 epoch...")
    
    best_val_loss = float('inf')
    loss_history = []
    
    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
        model.train()
        total_train_loss = 0.0
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss, val_wer = run_validation(model, processor, valid_loader, device)
        
        print(f"\n--- Epoch {epoch + 1} 總結 ---")
        print(f"  平均訓練 Loss: {avg_train_loss:.4f}")
        print(f"  驗證 Loss: {avg_val_loss:.4f}")
        print(f"  驗證 WER: {val_wer * 100:.2f} %")
        
        epoch_data = {
            'epoch': epoch + 1, 'train_loss': avg_train_loss,
            'validation_loss': avg_val_loss, 'validation_wer': val_wer
        }
        loss_history.append(epoch_data)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"❗️ 新的最佳驗證 Loss。正在儲存模型到 {OUTPUT_MODEL_DIR}")
            model.save_pretrained(OUTPUT_MODEL_DIR)
            processor.save_pretrained(OUTPUT_MODEL_DIR)
        else:
            print("驗證 Loss 未改善。")

    print("\n--- 訓練完成 ---")
    df_log = pd.DataFrame(loss_history)
    df_log.set_index('epoch', inplace=True)
    df_log.to_csv(LOG_FILE_PATH)
    print(f"紀錄已儲存於 {LOG_FILE_PATH}")

if __name__ == "__main__":
    train_fsc()



--- 開始微調 UsefulSensors/moonshine-tiny ---
正在載入模型...
✅ 添加了新的 [PAD] token。Tokenizer size: 32768
設定 pad_token 為 eos_token: </s> (ID: 2)
使用模型預設的 eos_token_id: 2
使用 tokenizer 新設定的 pad_token_id: 2
使用設備: cuda
將在 23132 筆訓練資料上訓練 50 個 epoch...

--- Epoch 1 / 50 ---


Training Epoch 1:   0%|          | 0/723 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Training Epoch 1: 100%|██████████| 723/723 [01:07<00:00, 10.64it/s]



Running validation...


Validation:   0%|          | 0/98 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   1%|          | 1/98 [00:01<02:46,  1.71s/it]


📋 驗證樣本檢查（第一個 batch 的前 3 筆）:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn the lights'
  WER: 25.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'off the lightshhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh'
  WER: 50.00%

樣本 3:
  REF: 'change language'
  HYP: 'language'
  WER: 50.00%



...


Running validation...


Validation:   0%|          | 0/98 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   1%|          | 1/98 [00:01<02:51,  1.77s/it]


📋 驗證樣本檢查（第一個 batch 的前 3 筆）:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'on lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights in lights inomely inomely inomely inomely inomely inomely inomely inomely inomely inomely inomely inomely inomely inumsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuitsuits'
  WER: 1325.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'turn the off lights the in theroom the off the lights the off the lights the in theroom the off the lights th

Validation: 100%|██████████| 98/98 [02:27<00:00,  1.51s/it]



--- Epoch 50 總結 ---
  平均訓練 Loss: 0.2755
  驗證 Loss: 0.4077
  驗證 WER: 1983.33 %
驗證 Loss 未改善。

--- 訓練完成 ---
紀錄已儲存於 training_log_simplified.csv


In [29]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
VALID_CSV = "data/valid_data.csv"
MODEL_NAME = "UsefulSensors/moonshine-tiny"
NUM_SAMPLES_TO_PREDICT = 3000
# -----------------

def normalize_text(text):
    """
    標準化文字：轉為小寫並移除標點符號，以便公平比較。
    """
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class FluentSpeechDataset(Dataset):
    """
    從 CSV 檔案載入資料的 Dataset。
    """
    def __init__(self, csv_path, fsc_base_path):
        self.df = pd.read_csv(csv_path)
        self.fsc_base_path = fsc_base_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['path']
        reference_text = row['transcription']
        full_audio_path = os.path.join(self.fsc_base_path, relative_path)
        audio_array, _ = librosa.load(full_audio_path, sr=16000)
        # 返回音訊和原始文本
        return {"audio": audio_array, "text": reference_text}

class DataCollatorForPrediction:
    """
    一個簡單的 DataCollator，用於準備預測用的批次資料。
    它會對音訊進行 padding，並將參考文字作為列表傳遞。
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_audios = [f["audio"] for f in features]
        reference_texts = [f["text"] for f in features]

        # 處理音訊輸入，自動 padding
        inputs = self.processor(
            audio=input_audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        batch = {}
        batch["input_features"] = inputs.input_values
        batch["reference_texts"] = reference_texts # 將文字列表直接放入 batch 中
        return batch

def predict_validation_samples_with_dataloader():
    """
    載入原始預訓練模型，使用 DataLoader 處理資料，並預測驗證集的前幾筆。
    """
    print(f"--- 開始使用原始預訓練模型 {MODEL_NAME} 進行預測 ---")
    csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

    if not os.path.exists(csv_path):
        print(f"錯誤：找不到驗證集 CSV 檔案 {csv_path}")
        return

    # 1. 載入原始模型和 Processor
    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval() # 設置為評估模式
    print(f"模型已載入到 {device}")
    
    # 2. 建立 Dataset, DataCollator 和 DataLoader
    dataset = FluentSpeechDataset(csv_path, FSC_DATASET_PATH)
    data_collator = DataCollatorForPrediction(processor)
    dataloader = DataLoader(
        dataset,
        batch_size=NUM_SAMPLES_TO_PREDICT,
        shuffle=False,
        collate_fn=data_collator
    )
    
    print(f"\n正在使用 DataLoader 預測 {VALID_CSV} 的前 {NUM_SAMPLES_TO_PREDICT} 筆資料...")
    print("="*70)

    # 3. 只從 DataLoader 取出第一個批次
    try:
        first_batch = next(iter(dataloader))
        
        # 將 Tensor 移動到 GPU
        input_features = first_batch["input_features"].to(device)
        reference_texts = first_batch["reference_texts"]

        # 4. 執行推論 (使用最陽春的 generate)
        generated_ids = model.generate(input_features)
        hypothesis_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # 5. 標準化並印出結果
        for i in range(len(reference_texts)):
            norm_ref = normalize_text(reference_texts[i])
            norm_hyp = normalize_text(hypothesis_texts[i])

            print(f"\n樣本 {i + 1}:")
            print(f"  REF (參考): '{norm_ref}'")
            print(f"  HYP (預測): '{norm_hyp}'")
            
    except Exception as e:
        print(f"處理批次時發生錯誤: {e}")
    
    print("="*70)
    print("\n預測完成。")


if __name__ == "__main__":
    predict_validation_samples_with_dataloader()



--- 開始使用原始預訓練模型 UsefulSensors/moonshine-tiny 進行預測 ---
正在載入模型...


模型已載入到 cuda

正在使用 DataLoader 預測 data/valid_data.csv 的前 3000 筆資料...

樣本 1:
  REF (參考): 'turn on the lights'
  HYP (預測): 'turn on the lights'

樣本 2:
  REF (參考): 'turn off the lights'
  HYP (預測): 'turn off the lights'

樣本 3:
  REF (參考): 'change language'
  HYP (預測): 'change language'

樣本 4:
  REF (參考): 'pause the music'
  HYP (預測): 'pause the music'

樣本 5:
  REF (參考): 'resume'
  HYP (預測): 'resume'

樣本 6:
  REF (參考): 'volume down'
  HYP (預測): 'lime down'

樣本 7:
  REF (參考): 'turn the lights on'
  HYP (預測): 'turn the lights on'

樣本 8:
  REF (參考): 'switch on the lights'
  HYP (預測): 'switch on the lights'

樣本 9:
  REF (參考): 'lights on'
  HYP (預測): 'lights on'

樣本 10:
  REF (參考): 'switch off the lights'
  HYP (預測): 'switch off the lights'

樣本 11:
  REF (參考): 'turn the lights off'
  HYP (預測): 'turn the lights off'

樣本 12:
  REF (參考): 'lights off'
  HYP (預測): 'lights off'

樣本 13:
  REF (參考): 'volume up'
  HYP (預測): 'volume up'

樣本 14:
  REF (參考): 'turn up the volume'
  HYP (預測): 'turn up the volu

# 直接fintune

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MoonshineForConditionalGeneration, AutoTokenizer
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
TRAIN_CSV = "data/train_data.csv"
VALID_CSV = "data/valid_data.csv"
MODEL_NAME = "UsefulSensors/moonshine-tiny"
OUTPUT_MODEL_DIR = "./moonshine-tiny-finetuned-fsc-simplified_3"
LOG_FILE_PATH = "training_log_simplified_3.csv"

# DataLoader 相關參數
BATCH_SIZE = 48
NUM_WORKERS = 4 # 若環境支援多執行緒，可設為 > 0

# 訓練參數
LEARNING_RATE = 2*1e-6
EPOCHS = 30
# -----------------

class FluentSpeechDataset(Dataset):
    def __init__(self, csv_path, fsc_base_path):
        self.df = pd.read_csv(csv_path)
        self.fsc_base_path = fsc_base_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['path']
        reference_text = row['transcription']
        full_audio_path = os.path.join(self.fsc_base_path, relative_path)
        audio_array, sampling_rate = librosa.load(full_audio_path, sr=16000)
        duration_sec = len(audio_array) / sampling_rate
        return {"audio": audio_array, "text": reference_text, "duration": duration_sec}

def normalize_text(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class DataCollatorSpeechSeq2Seq:
    """
    一個簡單的 DataCollator，用於準備預測用的批次資料。
    它會對音訊進行 padding，並將參考文字作為列表傳遞。

    """
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer


    def __call__(self, features):
        input_audios = [f["audio"] for f in features]
        label_texts = [f["text"] for f in features]
        durations = [f["duration"] for f in features]

        # 處理音訊輸入，自動 padding
        inputs = self.processor(
            audio=input_audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )
        labels = self.tokenizer(
            label_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        labels_input_ids = labels.input_ids.clone()
        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels_input_ids[labels_input_ids == pad_token_id] = -100
        
        batch = {}
        batch["input_values"] = inputs.input_values
        batch["labels"] = labels_input_ids
        batch["durations"] = torch.tensor(durations, dtype=torch.float32)
        return batch


def run_validation(model, processor, valid_loader, device):
    print("\nRunning validation...")
    model.eval()
    total_val_loss = 0.0
    all_references = []
    all_hypotheses = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(valid_loader, desc="Validation")):
            durations = batch.pop("durations")
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # 計算 Loss
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
            
            max_duration_in_batch = torch.max(durations).item()
            max_new_tokens_limit = int(max_duration_in_batch * 6)


            # 生成預測文字 (使用模型的預設 generation config)
            
            generated_ids = model.generate(
                batch["input_values"],
                # num_beams=5,                 # 主要策略：Beam Search
                max_new_tokens=max_new_tokens_limit, # 安全網：長度限制
                # no_repeat_ngram_size=3,      # 輔助：防止 n-gram 重複
                # early_stopping=True          # 優化：提早停止
            )
            hypotheses = processor.batch_decode(generated_ids, skip_special_tokens=True)
            
            # 解碼參考文字
            labels = batch["labels"]
            labels[labels == -100] = processor.tokenizer.pad_token_id
            references = processor.batch_decode(labels, skip_special_tokens=True)

            # 正規化文字
            batch_hypotheses = [normalize_text(h) for h in hypotheses]
            batch_references = [normalize_text(r) for r in references]

            # ✅ 在第一個 batch 結束時，印出前 3 筆結果以供檢查
            if batch_idx == 0:
                print("\n" + "="*70)
                print("📋 驗證樣本檢查（第一個 batch 的前 3 筆）:")
                print("="*70)
                for i in range(min(3, len(batch_references))):
                    ref = batch_references[i]
                    hyp = batch_hypotheses[i]
                    print(f"\n樣本 {i+1}:")
                    print(f"  REF: '{ref}'")
                    print(f"  HYP: '{hyp}'")
                    # 計算單一樣本的 WER
                    if ref and hyp:
                        try:
                            sample_wer = jiwer.wer([ref], [hyp])
                            print(f"  WER: {sample_wer*100:.2f}%")
                        except Exception:
                            print("  WER: 計算失敗")
                print("="*70 + "\n")


            all_hypotheses.extend(batch_hypotheses)
            all_references.extend(batch_references)

    avg_val_loss = total_val_loss / len(valid_loader)
    
    # 過濾掉空的參考或預測，避免 jiwer 出錯
    valid_pairs = [(ref, hyp) for ref, hyp in zip(all_references, all_hypotheses) if ref and hyp]
    if not valid_pairs:
        print("警告：沒有有效的樣本對可以計算 WER！")
        val_wer = 1.0
    else:
        filtered_references, filtered_hypotheses = zip(*valid_pairs)
        val_wer = jiwer.wer(list(filtered_references), list(filtered_hypotheses))
    
    model.train()
    return avg_val_loss, val_wer

def train_fsc():
    print(f"--- 開始微調 {MODEL_NAME} ---")
    train_csv_path = os.path.join(FSC_DATASET_PATH, TRAIN_CSV)
    valid_csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

    # 1. 載入模型和 Processor
    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)

    tokenizer = processor.tokenizer
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
    tokenizer.eos_token = '</s>'
    model.generation_config.pad_token_id = model.config.pad_token_id
    print(f"設定 pad_token : {tokenizer.pad_token} (ID: {model.generation_config.pad_token_id})")
    print(f"設定 eos_token : {tokenizer.eos_token} (ID: {model.generation_config.eos_token_id})")

    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"使用設備: {device}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    # 2. 建立 Dataset 和 DataLoader
    train_dataset = FluentSpeechDataset(train_csv_path, FSC_DATASET_PATH)
    print("檢查訓練資料樣本:")
    for i in range(5):
        print(f"{i+1}. {train_dataset[i]['text']}")
    valid_dataset = FluentSpeechDataset(valid_csv_path, FSC_DATASET_PATH)
    
    data_collator = DataCollatorSpeechSeq2Seq(processor)
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    
    print(f"將在 {len(train_dataset)} 筆訓練資料上訓練 {EPOCHS} 個 epoch...")
    
    best_val_loss = float('inf')
    loss_history = []
    
    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
        model.train()
        total_train_loss = 0.0
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss, val_wer = run_validation(model, processor, valid_loader, device)
        
        print(f"\n--- Epoch {epoch + 1} 總結 ---")
        print(f"  平均訓練 Loss: {avg_train_loss:.4f}")
        print(f"  驗證 Loss: {avg_val_loss:.4f}")
        print(f"  驗證 WER: {val_wer * 100:.2f} %")
        
        epoch_data = {
            'epoch': epoch + 1, 'train_loss': avg_train_loss,
            'validation_loss': avg_val_loss, 'validation_wer': val_wer
        }
        loss_history.append(epoch_data)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"❗️ 新的最佳驗證 Loss。正在儲存模型到 {OUTPUT_MODEL_DIR}")
            model.save_pretrained(OUTPUT_MODEL_DIR)
            processor.save_pretrained(OUTPUT_MODEL_DIR)
        else:
            print("驗證 Loss 未改善。")

    print("\n--- 訓練完成 ---")
    df_log = pd.DataFrame(loss_history)
    df_log.set_index('epoch', inplace=True)
    df_log.to_csv(LOG_FILE_PATH)
    print(f"紀錄已儲存於 {LOG_FILE_PATH}")

if __name__ == "__main__":
    train_fsc()

--- 開始微調 UsefulSensors/moonshine-tiny ---
正在載入模型...
設定 pad_token : [PAD] (ID: 32768)
設定 eos_token : </s> (ID: 2)
使用設備: cuda
將在 23132 筆訓練資料上訓練 30 個 epoch...

--- Epoch 1 / 30 ---


Training Epoch 1:   0%|          | 0/482 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Training Epoch 1: 100%|██████████| 482/482 [00:50<00:00,  9.52it/s]



Running validation...


Validation:   0%|          | 0/65 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   3%|▎         | 2/65 [00:00<00:15,  4.11it/s]


📋 驗證樣本檢查（第一個 batch 的前 3 筆）:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn lights'
  WER: 50.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'turn lights'
  WER: 50.00%

樣本 3:
  REF: 'change language'
  HYP: 'change language'
  WER: 0.00%



...


Running validation...


Validation:   0%|          | 0/65 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   3%|▎         | 2/65 [00:00<00:17,  3.65it/s]


📋 驗證樣本檢查（第一個 batch 的前 3 筆）:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'on theetch up lamp lamp lamp lamp lamp lamp lamp lamp lamp'
  WER: 300.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'off lights'
  WER: 50.00%

樣本 3:
  REF: 'change language'
  HYP: 'language language language languages languages languages languages languages languages languages languages languages languages'
  WER: 600.00%



Validation: 100%|██████████| 65/65 [00:14<00:00,  4.55it/s]



--- Epoch 30 總結 ---
  平均訓練 Loss: 0.2978
  驗證 Loss: 0.4083
  驗證 WER: 238.00 %
❗️ 新的最佳驗證 Loss。正在儲存模型到 ./moonshine-tiny-finetuned-fsc-simplified_2

--- 訓練完成 ---
紀錄已儲存於 training_log_simplified_2.csv


# 凍結權重fintune

In [2]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
TRAIN_CSV = "data/train_data.csv"
VALID_CSV = "data/valid_data.csv"
MODEL_NAME = "UsefulSensors/moonshine-tiny"
OUTPUT_MODEL_DIR = "./moonshine-tiny-finetuned-fsc-frozen-encoder"
LOG_FILE_PATH = "training_log_frozen-encoder.csv"

# DataLoader 相關參數
BATCH_SIZE = 48
NUM_WORKERS = 4

# 訓練參數
LEARNING_RATE = 1e-5 # 可以用回稍高一點的學習率，因為可訓練參數變少了
EPOCHS = 10
# -----------------


class FluentSpeechDataset(Dataset):
    def __init__(self, csv_path, fsc_base_path):
        self.df = pd.read_csv(csv_path)
        self.fsc_base_path = fsc_base_path
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['path']
        reference_text = row['transcription']
        full_audio_path = os.path.join(self.fsc_base_path, relative_path)
        audio_array, sampling_rate = librosa.load(full_audio_path, sr=16000)
        duration_sec = len(audio_array) / sampling_rate
        return {"audio": audio_array, "text": reference_text, "duration": duration_sec}

def normalize_text(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

class DataCollatorSpeechSeq2Seq:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
    def __call__(self, features):
        input_audios = [f["audio"] for f in features]
        label_texts = [normalize_text(f["text"]) for f in features]
        durations = [f["duration"] for f in features]
        inputs = self.processor(audio=input_audios, sampling_rate=16000, return_tensors="pt", padding=True)
        labels = self.tokenizer(label_texts, padding=True, truncation=True, return_tensors="pt")
        labels_input_ids = labels.input_ids.clone()
        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels_input_ids[labels_input_ids == pad_token_id] = -100
        batch = {}
        batch["input_values"] = inputs.input_values
        batch["labels"] = labels_input_ids
        batch["durations"] = torch.tensor(durations, dtype=torch.float32)
        return batch

def run_validation(model, processor, valid_loader, device):
    print("\nRunning validation...")
    model.eval()
    total_val_loss = 0.0
    all_references = []
    all_hypotheses = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(valid_loader, desc="Validation")):
            durations = batch.pop("durations").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
            generated_ids = model.generate(
                batch["input_values"],
                num_beams=5,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
            hypotheses = processor.batch_decode(generated_ids, skip_special_tokens=True)
            labels = batch["labels"]
            labels[labels == -100] = processor.tokenizer.pad_token_id
            references = processor.batch_decode(labels, skip_special_tokens=True)
            batch_hypotheses = [normalize_text(h) for h in hypotheses]
            batch_references = [normalize_text(r) for r in references]
            if batch_idx == 0:
                print("\n" + "="*70)
                print("📋 驗證樣本檢查:")
                print("="*70)
                for i in range(min(3, len(batch_references))):
                    ref, hyp = batch_references[i], batch_hypotheses[i]
                    print(f"\n樣本 {i+1}:\n  REF: '{ref}'\n  HYP: '{hyp}'")
                    if ref and hyp: print(f"  WER: {jiwer.wer([ref], [hyp])*100:.2f}%")
                print("="*70 + "\n")
            all_hypotheses.extend(batch_hypotheses)
            all_references.extend(batch_references)
    avg_val_loss = total_val_loss / len(valid_loader)
    valid_pairs = [(r, h) for r, h in zip(all_references, all_hypotheses) if r and h]
    val_wer = 1.0 if not valid_pairs else jiwer.wer([p[0] for p in valid_pairs], [p[1] for p in valid_pairs])
    model.train()
    return avg_val_loss, val_wer


def train_fsc():
    print(f"--- 開始微調 (凍結 Encoder) ---")
    train_csv_path = os.path.join(FSC_DATASET_PATH, TRAIN_CSV)
    valid_csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
    tokenizer = processor.tokenizer
    
    # 修正 Tokenizer 的 EOS Token
    if tokenizer.eos_token is None:
        tokenizer.eos_token = '</s>'
    
    # 設定獨立的 PAD Token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))
    
    pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
    model.config.pad_token_id = pad_token_id
    model.generation_config.pad_token_id = pad_token_id
    
    # --- ❗️ 關鍵修正：凍結 Encoder 的所有參數 ---
    print("正在凍結 Encoder 的權重...")
    for param in model.model.encoder.parameters():
        param.requires_grad = False
    # ---------------------------------------------
    
    # 驗證凍結是否成功 (可選)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"模型總參數: {total_params / 1_000_000:.2f}M")
    print(f"可訓練參數: {trainable_params / 1_000_000:.2f}M")
    print(f"凍結比例: {(total_params - trainable_params) / total_params:.2%}")
    # ------------------------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"使用設備: {device}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    train_dataset = FluentSpeechDataset(train_csv_path, FSC_DATASET_PATH)
    valid_dataset = FluentSpeechDataset(valid_csv_path, FSC_DATASET_PATH)
    
    data_collator = DataCollatorSpeechSeq2Seq(processor)
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    
    print(f"將在 {len(train_dataset)} 筆訓練資料上訓練 {EPOCHS} 個 epoch...")
    
    best_val_wer = float('inf')
    loss_history = []
    
    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
        model.train()
        total_train_loss = 0.0
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")):
            batch.pop("durations", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss, val_wer = run_validation(model, processor, valid_loader, device)
        
        print(f"\n--- Epoch {epoch + 1} 總結 ---")
        print(f"  平均訓練 Loss: {avg_train_loss:.4f}")
        print(f"  驗證 Loss: {avg_val_loss:.4f}")
        print(f"  驗證 WER: {val_wer * 100:.2f} %")
        
        epoch_data = {
            'epoch': epoch + 1, 'train_loss': avg_train_loss,
            'validation_loss': avg_val_loss, 'validation_wer': val_wer
        }
        loss_history.append(epoch_data)
        
        if val_wer < best_val_wer:
            best_val_wer = val_wer
            print(f"❗️ 新的最佳驗證 WER。正在儲存模型到 {OUTPUT_MODEL_DIR}")
            model.save_pretrained(OUTPUT_MODEL_DIR)
            processor.save_pretrained(OUTPUT_MODEL_DIR)
        else:
            print("驗證 WER 未改善。")

    print("\n--- 訓練完成 ---")
    df_log = pd.DataFrame(loss_history)
    df_log.set_index('epoch', inplace=True)
    df_log.to_csv(LOG_FILE_PATH)
    print(f"紀錄已儲存於 {LOG_FILE_PATH}")

if __name__ == "__main__":
    train_fsc()

--- 開始微調 (凍結 Encoder) ---
正在載入模型...
正在凍結 Encoder 的權重...
模型總參數: 27.09M
可訓練參數: 19.41M
凍結比例: 28.35%
使用設備: cuda
將在 23132 筆訓練資料上訓練 10 個 epoch...

--- Epoch 1 / 10 ---


Training Epoch 1:   0%|          | 0/482 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Training Epoch 1: 100%|██████████| 482/482 [00:25<00:00, 18.63it/s]



Running validation...


Validation:   0%|          | 0/65 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   2%|▏         | 1/65 [00:11<12:37, 11.84s/it]


📋 驗證樣本檢查:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn the lights'
  WER: 25.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'off lights'
  WER: 50.00%

樣本 3:
  REF: 'change language'
  HYP: 'change language'
  WER: 0.00%



...


Running validation...


Validation:   0%|          | 0/65 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Validation:   2%|▏         | 1/65 [00:12<13:16, 12.44s/it]


📋 驗證樣本檢查:

樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn the on lights in bed theroom on the lights'
  WER: 150.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'turn the off lights in bed theh lights'
  WER: 125.00%

樣本 3:
  REF: 'change language'
  HYP: 'change language'
  WER: 0.00%



Validation: 100%|██████████| 65/65 [12:22<00:00, 11.43s/it]


--- Epoch 10 總結 ---
  平均訓練 Loss: 0.3550
  驗證 Loss: 0.4070
  驗證 WER: 205.98 %
驗證 WER 未改善。

--- 訓練完成 ---
紀錄已儲存於 training_log_frozen-encoder.csv


# 一筆一筆finetune

In [4]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"
TRAIN_CSV = "data/train_data.csv"
VALID_CSV = "data/valid_data.csv"
MODEL_NAME = "UsefulSensors/moonshine-tiny"
OUTPUT_MODEL_DIR = "./moonshine-tiny-finetuned-fsc-no-padding"
LOG_FILE_PATH = "training_log_no_padding.csv"

# --- ❗️ 關鍵修改 1：設定 DataLoader 和梯度累積 ---
# DataLoader 的 batch_size 設為 1，這樣就不需要 padding
BATCH_SIZE = 1 
# 我們希望模擬的「有效 batch size」
EFFECTIVE_BATCH_SIZE = 32 
# 計算梯度累積的步驟數
GRADIENT_ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // BATCH_SIZE

NUM_WORKERS = 4

# 訓練參數
LEARNING_RATE = 1e-5
EPOCHS = 5
# -----------------

class FluentSpeechDataset(Dataset):
    def __init__(self, csv_path, fsc_base_path):
        self.df = pd.read_csv(csv_path)
        self.fsc_base_path = fsc_base_path
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['path']
        reference_text = row['transcription']
        full_audio_path = os.path.join(self.fsc_base_path, relative_path)
        audio_array, sampling_rate = librosa.load(full_audio_path, sr=16000)
        duration_sec = len(audio_array) / sampling_rate
        return {"audio": audio_array, "text": reference_text, "duration": duration_sec}

def normalize_text(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- ❗️ 關鍵修改 2：DataCollator 不再需要處理 padding ---
class DataCollatorNoPadding:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
    
    def __call__(self, features):
        # 因為 batch size 是 1，features 列表裡永遠只有一個元素
        feature = features[0]
        input_audio = feature["audio"]
        label_text = normalize_text(feature["text"])
        duration = feature["duration"]

        # 處理單一音訊，padding=False
        inputs = self.processor(audio=input_audio, sampling_rate=16000, return_tensors="pt", padding=False)
        
        # 處理單一文字，padding=False
        labels = self.tokenizer(label_text, return_tensors="pt", padding=False)
        
        batch = {}
        batch["input_values"] = inputs.input_values
        batch["labels"] = labels.input_ids
        batch["durations"] = torch.tensor([duration], dtype=torch.float32) # 仍然需要是 Tensor
        
        return batch

def run_validation(model, processor, valid_loader, device):
    print("\nRunning validation...")
    model.eval()
    total_val_loss = 0.0
    all_references = []
    all_hypotheses = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(valid_loader, desc="Validation")):
            durations = batch.pop("durations").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
            generated_ids = model.generate(
                batch["input_values"],
                max_new_tokens=int(torch.max(durations).item() * 6),
                num_beams=1,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
            hypotheses = processor.batch_decode(generated_ids, skip_special_tokens=True)
            labels = batch["labels"]
            references = processor.batch_decode(labels, skip_special_tokens=True)
            batch_hypotheses = [normalize_text(h) for h in hypotheses]
            batch_references = [normalize_text(r) for r in references]
            if batch_idx < 3: # 只印出前三個樣本
                ref, hyp = batch_references[0], batch_hypotheses[0]
                print(f"\n樣本 {batch_idx+1}:\n  REF: '{ref}'\n  HYP: '{hyp}'")
                if ref and hyp: print(f"  WER: {jiwer.wer([ref], [hyp])*100:.2f}%")
            all_hypotheses.extend(batch_hypotheses)
            all_references.extend(batch_references)
    avg_val_loss = total_val_loss / len(valid_loader)
    valid_pairs = [(r, h) for r, h in zip(all_references, all_hypotheses) if r and h]
    val_wer = 1.0 if not valid_pairs else jiwer.wer([p[0] for p in valid_pairs], [p[1] for p in valid_pairs])
    model.train()
    return avg_val_loss, val_wer

def train_fsc():
    print(f"--- 開始微調 (無填充 & 梯度累積) ---")
    train_csv_path = os.path.join(FSC_DATASET_PATH, TRAIN_CSV)
    valid_csv_path = os.path.join(FSC_DATASET_PATH, VALID_CSV)

    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
    tokenizer = processor.tokenizer
    
    # 修正 Tokenizer 的 EOS Token
    if tokenizer.eos_token is None:
        tokenizer.eos_token = '</s>'
    
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"使用設備: {device}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    train_dataset = FluentSpeechDataset(train_csv_path, FSC_DATASET_PATH)
    valid_dataset = FluentSpeechDataset(valid_csv_path, FSC_DATASET_PATH)
    
    # 使用新的無填充 DataCollator
    data_collator = DataCollatorNoPadding(processor)
    
    # DataLoader 的 batch_size 設為 1
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=NUM_WORKERS, collate_fn=data_collator
    )
    
    print(f"將在 {len(train_dataset)} 筆訓練資料上訓練 {EPOCHS} 個 epoch...")
    print(f"梯度累積步驟: {GRADIENT_ACCUMULATION_STEPS} (有效批次大小: {EFFECTIVE_BATCH_SIZE})")
    
    best_val_wer = float('inf')
    loss_history = []
    
    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
        model.train()
        total_train_loss = 0.0
        
        # --- ❗️ 關鍵修改 3：實現梯度累積的訓練迴圈 ---
        optimizer.zero_grad() # 在迴圈開始前清零一次梯度
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")):
            # 訓練時不需要 durations，但我們讓它留在 batch 中也無妨
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            loss = outputs.loss
            
            # 將 loss 除以累積步驟數，以標準化 loss
            loss = loss / GRADIENT_ACCUMULATION_STEPS 
            
            loss.backward() # 計算梯度
            
            total_train_loss += loss.item()
            
            # 每 GRADIENT_ACCUMULATION_STEPS 次，才更新一次模型權重
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()    # 更新權重
                optimizer.zero_grad() # 清零梯度，為下一個累積週期做準備
        # --- 修改結束 ---
            
        avg_train_loss = total_train_loss / len(train_loader) * GRADIENT_ACCUMULATION_STEPS
        avg_val_loss, val_wer = run_validation(model, processor, valid_loader, device)
        
        print(f"\n--- Epoch {epoch + 1} 總結 ---")
        print(f"  平均訓練 Loss: {avg_train_loss:.4f}")
        print(f"  驗證 Loss: {avg_val_loss:.4f}")
        print(f"  驗證 WER: {val_wer * 100:.2f} %")
        
        epoch_data = { 'epoch': epoch + 1, 'train_loss': avg_train_loss, 'validation_loss': avg_val_loss, 'validation_wer': val_wer }
        loss_history.append(epoch_data)
        
        if val_wer < best_val_wer:
            best_val_wer = val_wer
            print(f"❗️ 新的最佳驗證 WER。正在儲存模型到 {OUTPUT_MODEL_DIR}")
            model.save_pretrained(OUTPUT_MODEL_DIR)
            processor.save_pretrained(OUTPUT_MODEL_DIR)
        else:
            print("驗證 WER 未改善。")

    print("\n--- 訓練完成 ---")
    df_log = pd.DataFrame(loss_history)
    df_log.set_index('epoch', inplace=True)
    df_log.to_csv(LOG_FILE_PATH)
    print(f"紀錄已儲存於 {LOG_FILE_PATH}")

if __name__ == "__main__":
    train_fsc()

--- 開始微調 (無填充 & 梯度累積) ---
正在載入模型...
使用設備: cuda
將在 23132 筆訓練資料上訓練 5 個 epoch...
梯度累積步驟: 32 (有效批次大小: 32)

--- Epoch 1 / 5 ---


Training Epoch 1: 100%|██████████| 23132/23132 [15:32<00:00, 24.81it/s]



Running validation...


Validation:   0%|          | 2/3118 [00:01<26:57,  1.93it/s]


樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn the lights'
  WER: 25.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'off lightsaut lightsautautauturautaut'
  WER: 75.00%

樣本 3:
  REF: 'change language'
  HYP: 'language'
  WER: 50.00%


...


Running validation...


Validation:   0%|          | 2/3118 [00:01<33:57,  1.53it/s]  


樣本 1:
  REF: 'turn on the lights'
  HYP: 'turn the on lights in bed lights in bath lights in'
  WER: 200.00%

樣本 2:
  REF: 'turn off the lights'
  HYP: 'off the lights theating hot theating up the'
  WER: 150.00%

樣本 3:
  REF: 'change language'
  HYP: 'language'
  WER: 50.00%


Validation: 100%|██████████| 3118/3118 [05:20<00:00,  9.74it/s]



--- Epoch 5 總結 ---
  平均訓練 Loss: 0.4147
  驗證 Loss: 0.5080
  驗證 WER: 136.17 %
驗證 WER 未改善。

--- 訓練完成 ---
紀錄已儲存於 training_log_no_padding.csv


讀取CSV 畫loss曲線 和 WER

In [ ]:
# 

# 測試

In [ ]:
import torch
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import os

# --- 設定最佳模型路徑 ---
MODEL_TO_CHECK_PATH = "./moonshine-tiny-finetuned-fsc-simplified_2" 
# ---------------------------------------------------



--- 正在檢查模型狀態：./moonshine-tiny-finetuned-fsc-simplified_2 ---

--- 1. Tokenizer 狀態 (Processor) ---
tokenizer.pad_token:      [PAD]
tokenizer.pad_token_id:   32768
tokenizer.eos_token:      None
tokenizer.eos_token_id:   None

--- 2. 模型核心設定 (Model Config) ---
model.config.pad_token_id:  32768
model.config.eos_token_id:  2

--- 3. 模型生成設定 (Generation Config) ---
model.generation_config.pad_token_id: 32768
model.generation_config.eos_token_id: 2

--- 健檢分析 ---
✅ PAD/EOS 設定正確且分離。
   - pad_token_id: 32768
   - eos_token_id: None
   問題可能出在學習率或資料本身。


In [11]:
import os
import pandas as pd
import torch
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import librosa
import jiwer
import re
from tqdm import tqdm

# --- 1. 設定 ---
# 設定 Fluent Speech Commands 資料集路徑
FSC_DATASET_PATH = "/path/to/fluent_speech_commands_dataset"

# 根據資料集 README，測試集的 CSV 路徑
TEST_CSV = "data/test_data.csv"

# 要評估的模型 (來自 README.md)
MODEL_NAME = "UsefulSensors/moonshine-tiny" # 或 "UsefulSensors/moonshine-base"
# -----------------

def normalize_text(text):
    """
    標準化文字：轉為小寫並移除標點符號，以便公平比較 WER。
    """
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def evaluate_fsc():
    print(f"--- 開始評估模型：{MODEL_NAME} ---")
    csv_path = os.path.join(FSC_DATASET_PATH, TEST_CSV)

    if not os.path.exists(csv_path):
        print(f"錯誤：找不到測試檔案 {csv_path}")
        return None, None # ❗️ 回傳 None 表示失敗


    print("正在載入模型...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = MoonshineForConditionalGeneration.from_pretrained(MODEL_NAME)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"模型已載入到 {device}")

    df = pd.read_csv(csv_path)
    
    results = []

    print(f"正在對 {TEST_CSV} 中的 {len(df)} 個音檔進行轉錄...")
    
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        try:
            relative_path = row['path']
            reference_text = row['transcription']
            
            full_audio_path = os.path.join(FSC_DATASET_PATH, relative_path)

            if not os.path.exists(full_audio_path):
                continue

            audio_array, _ = librosa.load(full_audio_path, sr=16000)

            inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt").to(device)
            
            generated_ids = model.generate(
                **inputs,
                num_beams=5,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
            
            hypothesis_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            results.append({
                "path": relative_path,
                "reference": normalize_text(reference_text),
                "hypothesis": normalize_text(hypothesis_text)
            })

        except Exception as e:
            print(f"處理檔案 {relative_path} 時發生錯誤: {e}")

    if not results:
        print("沒有處理任何樣本，無法計算 WER。")
        return [], [] # ❗️ 回傳空列表

    all_references = [res["reference"] for res in results]
    all_hypotheses = [res["hypothesis"] for res in results]

    print("\n計算總 WER (字詞錯誤率)...")
    total_wer = jiwer.wer(all_references, all_hypotheses)
    
    for res in results:
        if not res["reference"]:
            res["wer"] = 1.0 if res["hypothesis"] else 0.0
        else:
            res["wer"] = jiwer.wer(res["reference"], res["hypothesis"])

    errors_high = [res for res in results if res["wer"] > 0.5]
    correct_zero = [res for res in results if res["wer"] == 0]

    print("\n" + "="*80)
    print("📋 評估結果分析")
    print("="*80)

    print(f"\n--- ✅ 完全正確 (WER = 0%) 的樣本 ({len(correct_zero)} 筆) ---")
    for i, res in enumerate(correct_zero[:10]):
        # --- ❗️ 新增內容：顯示 path ---
        print(f"  {i+1}. Path: {res['path']}")
        print(f"     REF: '{res['reference']}'")
        print(f"     HYP: '{res['hypothesis']}'")
    if len(correct_zero) > 10:
        print("     ...")

    print(f"\n--- ❌ 嚴重錯誤 (WER > 50%) 的樣本 ({len(errors_high)} 筆) ---")
    for i, res in enumerate(errors_high[:10]):
        # --- ❗️ 新增內容：顯示 path ---
        print(f"  {i+1}. [{res['wer']*100:.0f}%] Path: {res['path']}")
        print(f"           REF: '{res['reference']}'")
        print(f"           HYP: '{res['hypothesis']}'")
    if len(errors_high) > 10:
        print("           ...")

    print("\n" + "="*80)
    print("\n--- 📊 評估總結 ---")
    print(f"模型: {MODEL_NAME}")
    print(f"資料集: Fluent Speech Commands (Test Set)")
    print(f"總共處理音檔數量: {len(results)}")
    print(f"\nWord Error Rate (WER): {total_wer * 100:.2f} %")
    print("(WER 越低越好)")
    
    # --- ❗️ 新增內容：回傳兩個列表 ---
    return correct_zero, errors_high
    # ------------------------------------

if __name__ == "__main__":
    # --- ❗️ 新增內容：接收函式回傳的列表 ---
    correct_list, error_list = evaluate_fsc()

--- 開始評估模型：UsefulSensors/moonshine-tiny ---
正在載入模型...
模型已載入到 cuda
正在對 data/test_data.csv 中的 3793 個音檔進行轉錄...


100%|██████████| 3793/3793 [05:09<00:00, 12.26it/s]



計算總 WER (字詞錯誤率)...

📋 評估結果分析

--- ✅ 完全正確 (WER = 0%) 的樣本 (2761 筆) ---
  1. Path: wavs/speakers/4BrX8aDqK2cLZRYl/cbdf5700-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'turn on the lights'
     HYP: 'turn on the lights'
  2. Path: wavs/speakers/4BrX8aDqK2cLZRYl/cff92500-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'turn off the lights'
     HYP: 'turn off the lights'
  3. Path: wavs/speakers/4BrX8aDqK2cLZRYl/d36722a0-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'change language'
     HYP: 'change language'
  4. Path: wavs/speakers/4BrX8aDqK2cLZRYl/d73086b0-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'pause the music'
     HYP: 'pause the music'
  5. Path: wavs/speakers/4BrX8aDqK2cLZRYl/da54f830-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'resume'
     HYP: 'resume'
  6. Path: wavs/speakers/4BrX8aDqK2cLZRYl/ddb1b7c0-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'volume down'
     HYP: 'volume down'
  7. Path: wavs/speakers/4BrX8aDqK2cLZRYl/e1501430-452c-11e9-b1e4-e5985dca719e.wav
     REF: 'turn the light

In [14]:
# error_list 中 wer = 1.0 的樣本
error_list_worst = [res for res in error_list if res["wer"] == 1.0]
error_list_worst

[{'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/49a02e30-452d-11e9-b1e4-e5985dca719e.wav',
  'reference': 'lamp on',
  'hypothesis': 'were born',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/572fb520-452d-11e9-b1e4-e5985dca719e.wav',
  'reference': 'lamp off',
  'hypothesis': 'land boss',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/dd2316e0-452d-11e9-b1e4-e5985dca719e.wav',
  'reference': 'bedroom lights off',
  'hypothesis': 'it reminds of',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/ec446bf0-452e-11e9-b1e4-e5985dca719e.wav',
  'reference': 'less heat',
  'hypothesis': 'lassie',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/eebd0d10-452e-11e9-b1e4-e5985dca719e.wav',
  'reference': 'heat down',
  'hypothesis': 'he dont',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/1b5c4790-4530-11e9-b1e4-e5985dca719e.wav',
  'reference': 'pause',
  'hypothesis': 'pass',
  'wer': 1.0},
 {'path': 'wavs/speakers/4BrX8aDqK2cLZRYl/a0ad75e0-4530-11e9-b

In [15]:
len(error_list_worst)

83

計算 WER (字詞錯誤率)...

--- 評估完成 ---
模型: ./moonshine/moonshine-tiny-finetuned-fsc-no-padding
資料集: Fluent Speech Commands (Test Set)
總共處理音檔數量: 3793

Word Error Rate (WER): 222.62 %
(WER 越低越好)